# Transformers — Runnable Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeevchiran/learnings-ai-ml/blob/main/notebook/transformers/transformers-lab.ipynb)

Companion notebook for the **Transformers** track (`trf-m1` … `trf-m9`). Builds
self-attention and multi-head attention from scratch, checks them against
PyTorch's own `scaled_dot_product_attention`, verifies the sinusoidal
positional encoding, and trains a tiny decoder-only transformer on a toy
task end to end — small enough to watch the loss actually drop on CPU.

Everything runs on **CPU in well under a minute total**. No dataset downloads.

| Part | Modules | What runs |
|---|---|---|
| 1 | trf-m4 – trf-m5 | Query/Key/Value by hand, causal masking, the Δembedding update |
| 2 | trf-m6 – trf-m7 | multi-head attention, checked against torch's own implementation |
| 3 | trf-m8 | sinusoidal positional encoding, verified |
| 4 | trf-m9 | a tiny GPT-style model, trained end to end |

## Setup

In [ ]:
import importlib.util, subprocess, sys

for pkg, module in [('torch', 'torch'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(module) is None:
        print(f'installing {pkg} …')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    else:
        print(f'{pkg:<12} already present')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math
import matplotlib.pyplot as plt

torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
plt.rcParams['figure.figsize'] = (7, 4)
print('torch  ', torch.__version__)
print('device ', DEVICE)

---
# Part 1 — Self-attention by hand

## trf-m4 · Query, Key, and the causal mask

The same toy sentence the modules use: "a fluffy blue creature roamed the
verdant forest". Hand-picked 2-D Query/Key/Value vectors — small enough to
verify every number, in the same spirit as every other track's notebook.

In [ ]:
TOKENS = ['a', 'fluffy', 'blue', 'creature', 'roamed', 'the', 'verdant', 'forest']

# Same toy vectors the QueryKeyWidget and ValueUpdateWidget use.
Q = torch.tensor([[0,0],[0,0.2],[0,0.2],[1.0,0],[0,1.0],[0,0],[0,0.2],[1.0,0]])
K = torch.tensor([[0.10,0.10],[1.00,0.10],[0.90,0.15],[0.10,1.00],[0.20,0.20],[0.05,0.05],[0.95,0.10],[0.15,0.95]])
V = torch.tensor([[0.0,0.0],[0.9,0.1],[0.7,0.15],[0.15,0.9],[0.1,0.1],[0.0,0.0],[0.85,0.1],[0.15,0.85]])

def attend(i, causal=True):
    scores = Q[i] @ K.T                                 # (8,)
    if causal:
        scores = scores.clone()
        scores[i+1:] = float('-inf')
    alpha = F.softmax(scores, dim=0)
    context = alpha @ V
    return alpha, context

i = TOKENS.index('creature')
alpha, context = attend(i)
print('attention weights for "creature":')
for t, a in zip(TOKENS, alpha.tolist()):
    print(f'  {t:<10} {a:.3f}')
print('\ncontext vector (weighted sum of V):', context.tolist())
assert abs(alpha.sum().item() - 1.0) < 1e-6
assert alpha[i+1:].sum().item() == 0.0, 'causal mask should zero out every future position'

In [ ]:
# trf-m4's -inf claim, checked directly: masked scores must be exactly -inf pre-softmax,
# and exactly 0 post-softmax — not just small.
scores = Q[i].clone() @ K.T
scores_masked = scores.clone()
scores_masked[i+1:] = float('-inf')
print('raw masked scores:', scores_masked[i+1:].tolist())
assert all(s == float('-inf') for s in scores_masked[i+1:].tolist())

alpha_masked = F.softmax(scores_masked, dim=0)
print('post-softmax weights for masked positions:', alpha_masked[i+1:].tolist())
assert all(a == 0.0 for a in alpha_masked[i+1:].tolist())

## trf-m5 · The embedding update, Δe = Σ αⱼ vⱼ

In [ ]:
E = torch.tensor([[0.05,0.05],[0.5,0.3],[0.4,0.35],[0.2,0.5],[0.3,0.1],[0.05,0.05],[0.45,0.32],[0.2,0.48]])

before = E[i]
_, delta = attend(i)
after = before + delta

print(f'E("creature") before : {before.tolist()}')
print(f'delta e              : {delta.tolist()}')
print(f'E("creature") after  : {after.tolist()}')
assert torch.allclose(after, before + delta)
print('\nThe update is additive — original meaning plus a context-dependent correction, never a replacement.')

---
# Part 2 — Multi-head attention, checked against PyTorch

## trf-m6 / trf-m7 · SelfAttentionHead and MultiHeadAttention

In [ ]:
class SelfAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.Wq = nn.Linear(embed_dim, head_dim, bias=False)
        self.Wk = nn.Linear(embed_dim, head_dim, bias=False)
        self.Wv = nn.Linear(embed_dim, head_dim, bias=False)
        self.head_dim = head_dim

    def forward(self, x, causal=True):
        Qh, Kh, Vh = self.Wq(x), self.Wk(x), self.Wv(x)
        scores = Qh @ Kh.transpose(-2, -1) / math.sqrt(self.head_dim)
        if causal:
            seq_len = x.size(1)
            mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
            scores = scores.masked_fill(mask, float('-inf'))
        alpha = F.softmax(scores, dim=-1)
        return alpha @ Vh

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList([SelfAttentionHead(embed_dim, head_dim) for _ in range(num_heads)])
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, causal=True):
        out = torch.cat([h(x, causal) for h in self.heads], dim=-1)
        return self.out_proj(out)

mha = MultiHeadAttention(embed_dim=512, num_heads=8)
x = torch.randn(2, 6, 512)
out = mha(x, causal=True)
print('output shape:', tuple(out.shape))
assert out.shape == x.shape

In [ ]:
# Sanity check ONE head's math against torch's own fused implementation.
head = SelfAttentionHead(embed_dim=32, head_dim=32)
x_small = torch.randn(1, 5, 32)

our_out = head(x_small, causal=True)

Qh, Kh, Vh = head.Wq(x_small), head.Wk(x_small), head.Wv(x_small)
torch_out = F.scaled_dot_product_attention(Qh, Kh, Vh, is_causal=True)

max_diff = (our_out - torch_out).abs().max().item()
print(f'max difference vs F.scaled_dot_product_attention: {max_diff:.2e}')
assert max_diff < 1e-5

---
# Part 3 — Positional encoding

## trf-m8 · The sinusoidal formula, verified

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(seq_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = sinusoidal_positional_encoding(seq_len=50, d_model=64)
print('shape:', tuple(pe.shape))

# Every position's encoding should be unique.
uniq = torch.unique(pe, dim=0)
print(f'{uniq.shape[0]} / {pe.shape[0]} positions have a unique encoding')
assert uniq.shape[0] == pe.shape[0]

plt.imshow(pe.T, cmap='RdBu', aspect='auto')
plt.xlabel('position'); plt.ylabel('encoding dimension'); plt.title('Sinusoidal positional encoding')
plt.colorbar(); plt.show()

---
# Part 4 — A tiny transformer, trained end to end

## trf-m9 · The reverse-sequence task, revisited with a transformer

Same toy task the RNN track's notebook used — reverse a sequence of digits —
now with a decoder-only transformer instead of an RNN, so the two notebooks
are directly comparable.

In [ ]:
SOS, EOS = 10, 11
VOCAB_SIZE = 12

def make_batch(batch_size, seq_len):
    src = torch.randint(0, 10, (batch_size, seq_len))
    tgt_body = src.flip(dims=[1])
    sos_col = torch.full((batch_size, 1), SOS)
    eos_col = torch.full((batch_size, 1), EOS)
    return src, torch.cat([sos_col, tgt_body, eos_col], dim=1)

class TinyTransformerLM(nn.Module):
    """Decoder-only: concatenate src and tgt, predict next token throughout."""
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2, max_len=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.register_buffer('pos_enc', sinusoidal_positional_encoding(max_len, embed_dim))
        self.blocks = nn.ModuleList([
            nn.ModuleDict({'attn': MultiHeadAttention(embed_dim, num_heads), 'ff': nn.Sequential(
                nn.Linear(embed_dim, 4 * embed_dim), nn.GELU(), nn.Linear(4 * embed_dim, embed_dim))})
            for _ in range(num_layers)])
        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, tokens):
        x = self.embed(tokens) + self.pos_enc[:tokens.size(1)]
        for block in self.blocks:
            x = x + block['attn'](x, causal=True)
            x = x + block['ff'](x)
        return self.head(self.ln(x))

model = TinyTransformerLM(VOCAB_SIZE).to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
SEQ_LEN = 6

losses = []
for step in range(300):
    src, tgt = make_batch(64, SEQ_LEN)
    full = torch.cat([src, tgt], dim=1).to(DEVICE)          # decoder-only: one long sequence
    logits = model(full[:, :-1])
    # only the target portion (after src) counts toward the loss
    target_logits = logits[:, SEQ_LEN:]
    target_labels = full[:, SEQ_LEN + 1:]
    loss = F.cross_entropy(target_logits.reshape(-1, VOCAB_SIZE), target_labels.reshape(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f'step {step:>4}  loss {loss.item():.4f}')

plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss'); plt.title('Tiny transformer learning to reverse a sequence'); plt.show()
assert losses[-1] < losses[0] / 3, 'loss should drop substantially over training'

In [ ]:
@torch.no_grad()
def generate(model, src, max_len=8):
    model.eval()
    tokens = torch.cat([src, torch.tensor([[SOS]])], dim=1).to(DEVICE)
    for _ in range(max_len):
        logits = model(tokens)
        next_tok = logits[:, -1].argmax(-1, keepdim=True)
        tokens = torch.cat([tokens, next_tok], dim=1)
        if next_tok.item() == EOS:
            break
    model.train()
    return tokens[0, src.size(1) + 1:-1].tolist()

for _ in range(5):
    src, _ = make_batch(1, SEQ_LEN)
    pred = generate(model, src)
    expected = src[0].flip(0).tolist()
    match = '✓' if pred == expected else '✗'
    print(f'src={src[0].tolist()}  predicted={pred}  expected={expected}  {match}')

---
# Part 5 — Pretrained models from Hugging Face

**Everything above runs offline. This part does not** — each cell downloads
pretrained weights the first time it runs, so it needs an internet connection
(fine on Colab; the models used here are small). Skip this part if you are
running without network access — nothing earlier depends on it.

In [ ]:
for pkg, module in [('transformers', 'transformers'), ('sentence-transformers', 'sentence_transformers')]:
    if importlib.util.find_spec(module) is None:
        print(f'installing {pkg} …')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    else:
        print(f'{pkg:<22} already present')

## trf-m11 · BERT tokenisation, and what the attention mask is for

The four preparation steps from the module — subword split, integer IDs,
special boundary tokens, attention mask — are all visible in one call.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('bert-base-uncased')

sentences = ['a fluffy blue creature roamed the verdant forest', 'short one']
batch = tok(sentences, padding=True, truncation=True, max_length=32, return_tensors='pt')

for s in sentences:
    print(f'{s!r}')
    print(f'  subword tokens: {tok.tokenize(s)}')

print('\ninput_ids shape:', tuple(batch['input_ids'].shape))
print('attention_mask:')
print(batch['attention_mask'])
print('\ndecoded row 1 (note the boundary tokens and the padding):')
print(' ', tok.decode(batch['input_ids'][1]))

# The mask is 1 for real tokens and 0 for padding — exactly the positions whose
# attention scores get driven to -inf, the same mechanism as trf-m4's causal mask.
n_real = batch['attention_mask'][1].sum().item()
n_total = batch['input_ids'].shape[1]
print(f'\nrow 1 has {n_real} real positions out of {n_total}')
assert batch['attention_mask'][0].sum() >= batch['attention_mask'][1].sum()

In [ ]:
from transformers import AutoModelForSequenceClassification

# A classification head on top of the pretrained encoder. The head starts random;
# fine-tuning updates it AND the encoder weights together.
clf = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

encoder_params = sum(p.numel() for p in clf.bert.parameters())
head_params = sum(p.numel() for p in clf.classifier.parameters())
print(f'pretrained encoder : {encoder_params:,} parameters')
print(f'classification head: {head_params:,} parameters')
print(f'the head is {head_params / encoder_params:.4%} of the encoder —')
print('almost all of the knowledge is already in the pretrained weights.')

with torch.no_grad():
    logits = clf(**batch).logits
print('\nlogits shape:', tuple(logits.shape), ' <- one score per class per sentence')
print('probabilities (head is untrained, so these are meaningless):')
print(F.softmax(logits, dim=-1))
assert logits.shape == (len(sentences), 2)

## trf-m12 · Translation (MarianMT) and question answering (T5)

In [ ]:
from transformers import pipeline

translator = pipeline('translation', model='Helsinki-NLP/opus-mt-en-fr')

for sentence in ['I love coffee.',
                 'A fluffy blue creature roamed the verdant forest.',
                 'The stock market closed lower today.']:
    out = translator(sentence, max_length=60)[0]['translation_text']
    print(f'EN: {sentence}')
    print(f'FR: {out}\n')

In [ ]:
# T5 is text-to-text: the task is described IN the input string, and the answer is
# GENERATED as a string — not located as start/end indices, which is the BERT-style
# extractive setup trf-m12 contrasts it against.
qa = pipeline('text2text-generation', model='google/flan-t5-base')

context = ('The transformer architecture was introduced in 2017 in the paper '
           'Attention Is All You Need. It replaced recurrence entirely with '
           'self-attention, which allowed training to be parallelised across positions.')

for question in ['What did the transformer replace recurrence with?',
                 'In what year was the transformer introduced?']:
    prompt = f'question: {question} context: {context}'
    answer = qa(prompt, max_length=40)[0]['generated_text']
    print(f'Q: {question}')
    print(f'A: {answer}\n')

## trf-m13 · Sentence embeddings and cosine similarity

The bi-encoder claim, measured: paraphrases should score high and unrelated
sentences low — with each sentence encoded exactly once.

In [ ]:
from sentence_transformers import SentenceTransformer, util

sbert = SentenceTransformer('all-MiniLM-L6-v2')

corpus = ['A rabbit hops through the field.',
          'A bunny bounds across the meadow.',
          'The stock market closed lower today.',
          'Equity indices finished the day down.']

embeddings = sbert.encode(corpus, convert_to_tensor=True)
print('embedding shape:', tuple(embeddings.shape), ' <- one fixed vector per sentence\n')

sims = util.cos_sim(embeddings, embeddings)
header = ' ' * 44 + ''.join(f'{i:>8}' for i in range(len(corpus)))
print(header)
for i, s in enumerate(corpus):
    row = ''.join(f'{sims[i][j]:>8.3f}' for j in range(len(corpus)))
    print(f'{i}: {s:<40}' + row)

# The two paraphrase pairs should each beat any cross-topic pairing.
assert sims[0][1] > sims[0][2] and sims[0][1] > sims[0][3]
assert sims[2][3] > sims[2][0] and sims[2][3] > sims[2][1]
print('\nParaphrase pairs (0,1) and (2,3) score higher than any cross-topic pair.')

In [ ]:
# Why the bi-encoder matters: cost. Encoding is O(N); comparing afterward is nearly free.
import time

queries = corpus * 25          # 100 sentences

start = time.time()
embs = sbert.encode(queries, convert_to_tensor=True, show_progress_bar=False)
encode_time = time.time() - start

start = time.time()
_ = util.cos_sim(embs, embs)   # all 100 x 100 pairs at once
compare_time = time.time() - start

print(f'encoding {len(queries)} sentences once : {encode_time:.3f}s')
print(f'comparing all {len(queries) ** 2:,} pairs     : {compare_time:.4f}s')
print('\nA cross-encoder would need a full forward pass for every one of those pairs.')
assert compare_time < encode_time

---
## Where to go next

- **Real tokenisation.** Swap the toy digit vocabulary for a subword tokenizer
  (`tiktoken`, `sentencepiece`) and a real text corpus — nothing about the
  architecture above changes, only the vocabulary size and the data loader.
- **Bigger scale.** Increase `embed_dim`, `num_heads`, `num_layers`, and watch
  the parameter count from trf-m6's GPT-3 table stop feeling abstract.
- **Actually fine-tune the BERT.** Part 5 builds the classifier and shows the
  shapes but stops short of a training loop — plug it into the same optimiser
  pattern used in Part 4 with a small labelled dataset.
- **KV-caching.** The `generate` function above recomputes attention over the
  whole sequence at every step — real inference servers cache past Keys and
  Values so each new token only costs one incremental attention step.

Re-run with more `num_layers` or a longer `SEQ_LEN` and the loss curve and
final reversal accuracy will both change — but the shape of the architecture,
and the fact that a decoder-only transformer learns this task at all with no
recurrence, will not.